In [1]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
import matplotlib.pyplot as plt
import seaborn as sns

# Set visualization style
sns.set_theme(style="whitegrid")

In [2]:
# Load the dataset
# Replace '\t' with ',' if your CSV is comma-separated
df = pd.read_csv('/content/drive/MyDrive/Colab Notebooks/MAI391_Project/weatherHCM.csv', sep='\t')

# Display the first few rows and check info
display(df.head())
df.info()

,Date,Time,Weather,Temp,Feels,Wind,Gust,Rain,Humidity,Cloud,Pressure,Vis
0,"Thu 01, Jan 2009",00:00,Fog,23 °c,25 °c,9 km/h from NNW,15 km/h,0.0 mm,97%,100%,1010 mb,Poor
1,"Thu 01, Jan 2009",03:00,Light drizzle,22 °c,25 °c,9 km/h from NNW,13 km/h,0.4 mm,97%,84%,1010 mb,Poor
2,"Thu 01, Jan 2009",06:00,Fog,22 °c,25 °c,6 km/h from N,8 km/h,0.0 mm,98%,100%,1011 mb,Poor
3,"Thu 01, Jan 2009",09:00,Cloudy,27 °c,31 °c,6 km/h from NNE,7 km/h,0.1 mm,83%,64%,1011 mb,Excellent
4,"Thu 01, Jan 2009",12:00,Partly cloudy,28 °c,34 °c,3 km/h from NE,3 km/h,0.0 mm,76%,62%,1010 mb,Excellent


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 34976 entries, 0 to 34975
Data columns (total 12 columns):
 #   Column    Non-Null Count  Dtype 
---  ------    --------------  ----- 
 0   Date      34976 non-null  object
 1   Time      34976 non-null  object
 2   Weather   34976 non-null  object
 3   Temp      34976 non-null  object
 4   Feels     34976 non-null  object
 5   Wind      34976 non-null  object
 6   Gust      34976 non-null  object
 7   Rain      34976 non-null  object
 8   Humidity  34976 non-null  object
 9   Cloud     34976 non-null  object
 10  Pressure  34976 non-null  object
 11  Vis       34976 non-null  object
dtypes: object(12)
memory usage: 3.2+ MB


In [3]:
import re

# 1. Ensure 'Date' is a proper datetime object
df['Date'] = pd.to_datetime(df['Date'], errors='coerce')

# 2. Define the numerical columns
numeric_cols = ['Temp', 'Feels', 'Wind', 'Gust', 'Rain', 'Humidity', 'Cloud', 'Pressure', 'Vis']

# 3. CLEANING STEP: Strip units (like °C, km/h, %) and convert to numeric
for col in numeric_cols:
    # Convert to string, remove everything except numbers, dots, and minus signs
    df[col] = df[col].astype(str).str.replace(r'[^0-9.-]', '', regex=True)
    # Convert to float
    df[col] = pd.to_numeric(df[col], errors='coerce')

# 4. Fill any actual missing values with the column mean (just in case)
df[numeric_cols] = df[numeric_cols].fillna(df[numeric_cols].mean())

# 5. Aggregate hourly data to daily averages
daily_numeric_df = df.groupby('Date')[numeric_cols].mean().reset_index()

# 6. Aggregate Weather mode
daily_weather = df.groupby('Date')['Weather'].agg(lambda x: x.mode()[0] if not x.mode().empty else "Unknown").reset_index()

# 7. Combine
df_processed = pd.merge(daily_numeric_df, daily_weather, on='Date')
print("Missing values after cleaning:", df_processed.isnull().sum().sum())

Missing values after cleaning: 4372


In [4]:
# 1. Convert the date field into a month feature to capture seasonal patterns [cite: 60, 109]
df_processed['Month'] = df_processed['Date'].dt.month

# 2. Encode categorical variables into numerical labels for classification [cite: 61, 111]
# We use pandas categorical codes to turn text like "Clear sky" into numbers (0, 1, 2...)
df_processed['Weather_Label'] = df_processed['Weather'].astype('category').cat.codes

# 3. Generate next-day prediction targets by shifting the weather condition by one time step [cite: 64, 112]
# This means features from day 't' will predict the label of day 't+1' [cite: 67, 229]
df_processed['Target_Next_Day_Label'] = df_processed['Weather_Label'].shift(-1)

# Drop the last row since it won't have a next-day target to predict
df_processed = df_processed.dropna(subset=['Target_Next_Day_Label'])

display(df_processed[['Date', 'Weather', 'Weather_Label', 'Target_Next_Day_Label']].head())

,Date,Weather,Weather_Label,Target_Next_Day_Label
0,2009-01-01,Cloudy,1,1.0
1,2009-01-02,Cloudy,1,13.0
2,2009-01-03,Partly cloudy,13,13.0
3,2009-01-04,Partly cloudy,13,13.0
4,2009-01-05,Partly cloudy,13,17.0


In [5]:
# Initialize the StandardScaler
scaler = StandardScaler()

# Apply Z-score normalization to the continuous features
df_processed[numeric_cols] = scaler.fit_transform(df_processed[numeric_cols])

print("Data after Z-score normalization:")
display(df_processed[numeric_cols].head())

Data after Z-score normalization:


/usr/local/lib/python3.12/dist-packages/sklearn/utils/extmath.py:1101: RuntimeWarning: invalid value encountered in divide
  updated_mean = (last_sum + new_sum) / updated_sample_count
/usr/local/lib/python3.12/dist-packages/sklearn/utils/extmath.py:1106: RuntimeWarning: invalid value encountered in divide
  T = new_sum / new_sample_count
/usr/local/lib/python3.12/dist-packages/sklearn/utils/extmath.py:1126: RuntimeWarning: invalid value encountered in divide
  new_unnormalized_variance -= correction**2 / new_sample_count


,Temp,Feels,Wind,Gust,Rain,Humidity,Cloud,Pressure,Vis
0,-1.923132,-1.701444,-0.783004,-0.582932,0.266762,1.495584,1.884732,0.499568,NaN
1,-2.210531,-2.538639,0.064763,0.268309,-0.550304,0.869939,1.112285,0.800566,NaN
2,-3.144580,-3.323509,-0.967301,-0.789294,-0.213865,1.315109,1.356563,1.101563,NaN
3,-2.569781,-2.800262,-1.483333,-1.356788,-0.566325,0.845875,0.115367,0.981164,NaN
4,-1.707582,-1.858418,-1.262176,-1.253608,-0.566325,0.256325,-0.544844,0.740366,NaN


In [6]:
# Save to a new CSV file
output_filename = 'hcm_weather_preprocessed.csv'
df_processed.to_csv(output_filename, index=False)

print(f"Preprocessing complete! Data successfully saved to {output_filename}.")

Preprocessing complete! Data successfully saved to hcm_weather_preprocessed.csv.
